# Referencia- és kiterjesztett implementáció konzisztencia-ellenőrzése

Ez a notebook a **„A Crouch-féle szimulációs prevalencia-becslés kiterjesztése folytonos időbeli modellezésre”** című szakdolgozat validációs anyaga.  
A szakdolgozat a Crouch-féle incidencia–túlélés alapú Monte Carlo prevalenciabecslés elméleti keretére és a meglévő `rprev` referencia-implementációra épül. A központi feladat a referencia-eljárás több indexdátum együttes kezelésére szolgáló kiterjesztése, valamint ennek módszertani és numerikus validálása (lásd: 3–5. fejezet).

Ennek megfelelően a **referencia** implementáció definíció szerint egyetlen indexdátumhoz ad prevalencia-becslést. A több időpontra vonatkozó becslés problémáját önálló módszertani kérdésként nem tárgyalja, így több indexdátum esetén egymástól független, ismételt egyindexű futtatásokkal adható eredmény. A **kiterjesztett implementáció** ezzel szemben explicit módon több indexdátum együttes kezelésére készült, és a teljes indexrácsra konzisztens becslést ad egy közös futási keretben (lásd: 4. fejezet – referencia implementáció és a kiterjesztés tárgya; 5. fejezet – matematikai modell és csomagszintű illesztés).


## Validációs célok

A notebook célja annak ellenőrzése, hogy a referencia implementáció és a több indexdátumot kezelő kiterjesztett implementáció azonos paraméterezés és azonos indexdátumok mellett konzisztens, módszertanilag értelmezhető eredményeket ad-e. Ennek érdekében az alábbi vizsgálatokat végezzük.

A jelölésekben K az indexdátumok számát jelöli: K=1 egyetlen indexdátumot, K>1 több indexdátumot jelent.

1. **Összehasonlíthatóság K=1 esetben**  
Egyetlen indexdátum mellett a referencia és a kiterjesztett implementáció kimenetei közvetlenül összevethetők, ugyanakkor a módszertani és algoritmikus különbségek miatt szigorú numerikus azonosság nem elvárás.  
A kiterjesztett eljárás esetszintű küszöb-alapú státuszképzése, valamint a véletlenszám-felhasználás eltérő szervezése K=1 esetben is okozhat eltérést. A validáció itt azt vizsgálja, hogy ezek az eltérések a becslési bizonytalanság tartományában maradnak-e, és nem utalnak-e szisztematikus torzításra (lásd: 5. fejezet, különösen a numerikus operacionalizálás; 6. fejezet).

2. **Időbeli konzisztencia K>1 esetben**  
Több indexdátum mellett a vizsgálat fókusza az eltérések mértékének és mintázatának módszertani értelmezhetősége.  
A kiterjesztett implementáció futáson belül időben konzisztens státuszképzést alkalmaz, ezért természetes, hogy viselkedése eltér az indexenként külön futtatott referencia-megközelítéstől. A kérdés az, hogy ez az eltérés összhangban áll-e a kiterjesztés definiált működésével (lásd: 5. fejezet; 6. fejezet).


## Módszertani eltérések és értelmezés

A két implementáció futási szerkezete eltér, ezért a véletlenszám-generátor hívásainak száma és ütemezése sem azonos (lásd dolgozat 5. fejezet). Ennek következtében azonos seed (véletlengenerátor magja) beállítás mellett sem várható bitazonos kimenet.  
A megfigyelt különbségek akkor tekinthetők elfogadhatónak, ha:
- nagyságuk a becslési bizonytalansággal összemérhető,
- nem mutatnak tartós, egyirányú torzítást az indexdátumok mentén,
- és összhangban állnak a kiterjesztés ismert szerkezeti sajátosságaival (lásd: 6–7. fejezet).
- nem haladják meg azt a változékonysági mértéket, amelyet önmagában a seed megválasztása okoz az eljárás kimenetében.


## Notebook felépítése

1. Közös paraméterezés rögzítése.
2. A referencia implementáció futtatása K=1 és K>1 esetekre indexdátumonként, különálló hívásokkal.
3. A kiterjesztett implementáció futtatása K=1 és K>1 esetekre.
4. A K=1 és K>1 eseek célzott öszevetése és kiértékelése.
5. Eltérések esetén kiegészítő diagnosztikák futtatása az eltérések forrásának azonosítására.


In [ ]:
# Common configuration (shared across runs)
seed <- 123L
N_boot <- 1000L
num_years <- c(3, 5, 10, 13, 15)

# Index date grid (K = number of index dates)
K <- 3L
index_start <- as.Date("2012-01-07")
index_dates <- seq.Date(from = index_start, by = "year", length.out = K)

# Model + prevalence settings
inc_formula <- entrydate ~ sex
surv_formula <- survival::Surv(time, status) ~ age + sex
dist <- "weibull"
population_size <- 1e6
death_column <- "eventdate"
status_col <- "status"


In [ ]:
# Environment: load package under test + required dependencies + example data
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")

# Resolve repo root so the notebook runs from any working directory
repo_path <- tryCatch(
  trimws(system("git rev-parse --show-toplevel", intern = TRUE)),
  error = function(e) NA_character_
)
if (is.na(repo_path) || !nzchar(repo_path)) repo_path <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)

devtools::load_all(repo_path, quiet = TRUE)
message("Loaded rprev from: ", repo_path)

branch <- tryCatch(system(sprintf('git -C "%s" rev-parse --abbrev-ref HEAD', repo_path), intern = TRUE),
  error = function(e) NA_character_
)
commit <- tryCatch(system(sprintf('git -C "%s" rev-parse HEAD', repo_path), intern = TRUE),
  error = function(e) NA_character_
)
message("Loaded branch: ", branch)
message("Loaded commit: ", commit)

suppressPackageStartupMessages({
  library(lubridate)
  library(survival)
})

data(prevsim)
registry_start <- min(prevsim$entrydate, na.rm = TRUE)


In [19]:
# Setup note: shared parameters and data are defined in the first two cells.


ℹ Loading rprev
Loaded rprev from: C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext

Loaded branch: notebooks/analysis



## 1) DEV multi-index run
We run the dev implementation once with fixed `index_dates` and shared parameters.

**Observation (saved run):** K>1 estimates are stable and no warnings appear.

**Why this matters:** this run defines the *single?population, multi?index* behavior described in the thesis.
If this output is inconsistent across horizons, it indicates a logic or indexing error, not RNG noise.


In [2]:
set.seed(seed)
res <- prevalence(
  index_dates = index_dates,
  num_years_to_estimate = num_years,
  data = prevsim,
  inc_formula = inc_formula,
  surv_formula = surv_formula,
  dist = dist,
  population_size = population_size,
  death_column = death_column,
  registry_start_date = registry_start,
  N_boot = N_boot
)
res



Estimated prevalence:
 index_dates  3y  5y    10y    13y    15y
  2012-01-07 207 297 507.96 596.80 648.66
  2013-01-07 207 304 517.00 605.72 657.65
  2014-01-07 127 226 433.00 529.35 581.54
Prevalence rate (per 1e+05):
 index_dates   3y   5y  10y   13y   15y
  2012-01-07 20.7 29.7 50.8 59.68 64.87
  2013-01-07 20.7 30.4 51.7 60.57 65.77
  2014-01-07 12.7 22.6 43.3 52.94 58.15

### Summary output
**Observation (saved run):** summary prints index?by?horizon tables as expected.

We expect the summary to print one row per index date and one column per horizon.
If any horizon is missing or misaligned, it points to issues in estimate assembly or printing.


In [3]:
summary(res)



Prevalence 
~~~~~~~~~~
Estimated prevalence:
 index_dates  3y  5y    10y    13y    15y
  2012-01-07 207 297 507.96 596.80 648.66
  2013-01-07 207 304 517.00 605.72 657.65
  2014-01-07 127 226 433.00 529.35 581.54
Prevalence rate (per 1e+05):
 index_dates   3y   5y  10y   13y   15y
  2012-01-07 20.7 29.7 50.8 59.68 64.87
  2013-01-07 20.7 30.4 51.7 60.57 65.77
  2014-01-07 12.7 22.6 43.3 52.94 58.15

Registry Data
~~~~~~~~~~~~~
Index dates: 2012-01-07 to 2014-01-07 (K=3) 
Start date: 2003-01-07 
Overall incidence rate: 0.249  (per day, up to  16077 )
Counted prevalent cases:
 index_dates counted
  2012-01-07     475
  2013-01-07     517
  2014-01-07     472

Simulation
~~~~~~~~~~
Iterations: 1000 
Average incidence rate: 0.309 
P-values:
 index_dates  p_value
  2012-01-07       NA
  2013-01-07       NA
  2014-01-07 0.127658


## 2) CRAN rprev (single-index baseline)
We run CRAN **separately for each index date** (K=1 baseline).

**Observation (saved run):** CRAN prints standard output; used as comparison baseline.

**Baseline definition:** CRAN runs each index separately, so it re?simulates populations per index.
This is the reference behavior for K=1 in the original package.


In [4]:
if (!requireNamespace('callr', quietly = TRUE)) install.packages('callr')
if (!requireNamespace('rprev', quietly = TRUE)) install.packages('rprev')

cran_outputs <- callr::r(function(index_dates, num_years, registry_start, seed, N_boot,
                                  inc_formula, surv_formula, dist, population_size, death_column) {
  library(rprev)
  library(lubridate)
  library(survival)
  data(prevsim, package = 'rprev')

  index_dates <- as.Date(index_dates)
  registry_start <- as.Date(registry_start)

  out <- lapply(index_dates, function(idx) {
    set.seed(seed)
    prevalence(
      index = idx,
      num_years_to_estimate = num_years,
      data = prevsim,
      inc_formula = inc_formula,
      surv_formula = surv_formula,
      dist = dist,
      population_size = population_size,
      death_column = death_column,
      registry_start_date = registry_start,
      N_boot = N_boot
    )
  })

  outputs <- lapply(seq_along(out), function(i) {
    idx <- index_dates[i]
    c(
      paste0('=== CRAN rprev index: ', idx, ' ==='),
      capture.output(print(out[[i]])),
      '',
      capture.output(summary(out[[i]])),
      ''
    )
  })
  unlist(outputs)
}, args = list(index_dates = index_dates,
              num_years = num_years,
              registry_start = registry_start,
              seed = seed,
              N_boot = N_boot,
              inc_formula = inc_formula,
              surv_formula = surv_formula,
              dist = dist,
              population_size = population_size,
              death_column = death_column))

cat(paste(cran_outputs, collapse = '
'))



=== CRAN rprev index: 2012-01-07 ===
Estimated prevalence at 2012-01-07:
3 years: 207 (20.7 per 1e+05) 
5 years: 297 (29.7 per 1e+05) 
10 years: 508.16 (50.82 per 1e+05) 
13 years: 597.18 (59.72 per 1e+05) 
15 years: 649.07 (64.91 per 1e+05) 
[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

[[4]]
NULL

[[5]]
NULL


Prevalence 
~~~~~~~~~~
Estimated prevalence at 2012-01-07:
3 years: 207 (20.7 per 1e+05) 
5 years: 297 (29.7 per 1e+05) 
10 years: 508.16 (50.82 per 1e+05) 
13 years: 597.18 (59.72 per 1e+05) 
15 years: 649.07 (64.91 per 1e+05) 

Registry Data
~~~~~~~~~~~~~
Index date: 2012-01-07 
Start date: 2003-01-07 
Overall incidence rate: 0.304 
Counted prevalent cases: 475 

Simulation
~~~~~~~~~~
Iterations: 1000 
Average incidence rate: 0.273 
P-value: 0.6004299

=== CRAN rprev index: 2013-01-07 ===
Estimated prevalence at 2013-01-07:
3 years: 207 (20.7 per 1e+05) 
5 years: 304 (30.4 per 1e+05) 
10 years: 517 (51.7 per 1e+05) 
13 years: 606.82 (60.68 per 1e+05) 
15 years: 658.7 (65.87 per 1e+05) 

## 3) K=1 comparison (DEV vs CRAN)
We compare K=1 results for the first index date.

**Observation (saved run):** all horizons match exactly (abs_diff = 0).

If this comparison fails, the issue is *not* multi?index logic; it is in the shared core (models, simulation, or counted logic).


In [5]:
index_single <- index_dates[1]

extract_est_table <- function(res, years) {
  rows <- lapply(years, function(y) {
    item <- paste0('y', y)
    est <- res$estimates[[item]]
    abs_val <- est[['absolute.prevalence']]
    rate_name <- grep('^per[^.]*$', names(est), value=TRUE)
    rate_val <- if (length(rate_name) == 1) est[[rate_name]] else NA_real_
    data.frame(year = y, absolute = as.numeric(abs_val), rate = as.numeric(rate_val))
  })
  do.call(rbind, rows)
}



In [6]:
# K=1 run using local (dev) package
set.seed(seed)
res_dev_k1 <- prevalence(
  index_dates = index_single,
  num_years_to_estimate = num_years,
  data = prevsim,
  inc_formula = inc_formula,
  surv_formula = surv_formula,
  dist = dist,
  population_size = population_size,
  death_column = death_column,
  registry_start_date = registry_start,
  N_boot = N_boot
)
dev_table <- extract_est_table(res_dev_k1, num_years)
dev_table



year,absolute,rate
<dbl>,<dbl>,<dbl>
3,207.00,20.70
5,297.00,29.70
10,508.08,50.81
13,597.02,59.70
15,648.92,64.89


In [7]:
# K=1 run using CRAN rprev in a separate R session
if (!requireNamespace('callr', quietly = TRUE)) install.packages('callr')
if (!requireNamespace('rprev', quietly = TRUE)) install.packages('rprev')

cran_table <- callr::r(function(idx, years, seed, registry_start, N_boot, pop_size,
                                inc_formula, surv_formula, dist, death_column) {
  library(rprev)
  library(lubridate)
  library(survival)
  data(prevsim, package = 'rprev')

  set.seed(seed)
  res <- prevalence(
    index = idx,
    num_years_to_estimate = years,
    data = prevsim,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = pop_size,
    death_column = death_column,
    registry_start_date = registry_start,
    N_boot = N_boot
  )

  extract_est_table <- function(res, years) {
    rows <- lapply(years, function(y) {
      item <- paste0('y', y)
      est <- res$estimates[[item]]
      abs_val <- est[['absolute.prevalence']]
      rate_name <- grep('^per[^.]*$', names(est), value=TRUE)
      rate_val <- if (length(rate_name) == 1) est[[rate_name]] else NA_real_
      data.frame(year = y, absolute = as.numeric(abs_val), rate = as.numeric(rate_val))
    })
    do.call(rbind, rows)
  }

  extract_est_table(res, years)
}, args = list(idx = index_single,
              years = num_years,
              seed = seed,
              registry_start = registry_start,
              N_boot = N_boot,
              pop_size = population_size,
              inc_formula = inc_formula,
              surv_formula = surv_formula,
              dist = dist,
              death_column = death_column))
cran_table



year,absolute,rate
<dbl>,<dbl>,<dbl>
3,207.00,20.70
5,297.00,29.70
10,508.16,50.82
13,597.18,59.72
15,649.07,64.91


In [8]:
# Comparison table
compare_table <- merge(dev_table, cran_table, by = 'year', suffixes = c('_dev', '_cran'))
compare_table$abs_diff <- compare_table$absolute_dev - compare_table$absolute_cran
compare_table$rate_diff <- compare_table$rate_dev - compare_table$rate_cran
compare_table



year,absolute_dev,rate_dev,absolute_cran,rate_cran,abs_diff,rate_diff
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3,207.00,20.70,207.00,20.70,0.00,0.00
5,297.00,29.70,297.00,29.70,0.00,0.00
10,508.08,50.81,508.16,50.82,-0.08,-0.01
13,597.02,59.70,597.18,59.72,-0.16,-0.02
15,648.92,64.89,649.07,64.91,-0.15,-0.02


## 4) Prefix-effect demo (K=1 vs K>1 with expanding horizons)
We check whether adding extra horizons changes the 13y estimate.

**Observation (saved run):** 13y is consistent across K settings (differences ? 0).

This check ensures that adding extra horizons does **not** shift earlier horizons.
If it does, the aggregation is order?dependent, which would be a bug.


In [9]:
# Prefix-effect demo: K=1 vs K>1 with expanding max horizon
set.seed(seed)
years_13 <- c(13)
years_10_13 <- c(10, 13)
years_10_13_15 <- c(10, 13, 15)

res_k1_13 <- prevalence(
  index_dates = index_dates,
  num_years_to_estimate = years_13,
  data = prevsim,
  inc_formula = inc_formula,
  surv_formula = surv_formula,
  dist = dist,
  population_size = population_size,
  death_column = death_column,
  registry_start_date = registry_start,
  N_boot = N_boot
)

set.seed(seed)
res_k10_13 <- prevalence(
  index_dates = index_dates,
  num_years_to_estimate = years_10_13,
  data = prevsim,
  inc_formula = inc_formula,
  surv_formula = surv_formula,
  dist = dist,
  population_size = population_size,
  death_column = death_column,
  registry_start_date = registry_start,
  N_boot = N_boot
)

set.seed(seed)
res_k10_13_15 <- prevalence(
  index_dates = index_dates,
  num_years_to_estimate = years_10_13_15,
  data = prevsim,
  inc_formula = inc_formula,
  surv_formula = surv_formula,
  dist = dist,
  population_size = population_size,
  death_column = death_column,
  registry_start_date = registry_start,
  N_boot = N_boot
)

# Extract 13y estimates for comparison (first index date)
get_y <- function(res, y) {
  item <- paste0('y', y)
  as.numeric(res$estimates[[item]]$absolute.prevalence)[1]
}

data.frame(
  case = c('K=1 (13y only)', 'K>1 (10+13)', 'K>1 (10+13+15)'),
  y13 = c(get_y(res_k1_13, 13), get_y(res_k10_13, 13), get_y(res_k10_13_15, 13))
)



case,y13
<chr>,<dbl>
K=1 (13y only),597.16
K>1 (10+13),597.16
K>1 (10+13+15),596.80
